# Decision trees with imperfect information (videos #6 and #7)

Python version of *Decision Trees with Imperfect Information (#6)* and *Solving a Decision Tree with Imperfect Information (#7)*. In Radiant: **Load example > Imperfect information: a popup store**.

## Example 3

> You are the owner of Radycal Kicks and deciding whether to open a new store. You believe there is a 60% chance of the store being successful. Now you are offered to run a short-term popup store to better evaluate store performance in the district. Running a popup store helps identify whether or not the location will be successful. You believe that the popup test will correctly identify successful locations 80% of the time and correctly identify failures 90% of the time. Given the result of popup test is successful, what is the probability of a real store being successful?

In [1]:
import pyrsm as rsm
from IPython.display import Markdown


def show(tree, final=True):
    """Draw a decision tree (Mermaid; rendered by JupyterLab, VS Code, Quarto)."""
    return Markdown(f"```mermaid\n{tree.to_mermaid(final=final)}\n```")

Bayes' rule: P(S|+) = P(+|S) P(S) / P(+), with P(+) = P(+|S) P(S) + P(+|F) P(F).

In [2]:
p_s, p_pos_s, p_neg_f = 0.6, 0.8, 0.9
p_pos = p_pos_s * p_s + (1 - p_neg_f) * (1 - p_s)
p_s_pos = p_pos_s * p_s / p_pos
round(p_pos, 4), round(p_s_pos, 4)

(0.52, 0.9231)

A positive popup test raises the probability of success from 60% to **92.3%**.

## Example 4

> Continued from Example 1. Now the owner of the retail space offers you the ability to run a short-term popup store to better evaluate store performance in the district. Doing so will cost an additional $500,000 but allow you to acquire potentially helpful information. Running a popup store helps identify whether or not the location will be successful. You believe that the popup test will correctly identify successful locations 80% of the time and correctly identify failures 90% of the time. What is the optimal strategy? What is the EMV associated with your strategy?

The same calculations as formulas in the `variables:` section: a formula can use any variable defined above it, and the tree uses the results.

In [3]:
full = """name: open new store
variables:
    fixed cost: 2000000
    popup cost: 500000
    P(S): 0.6
    P(F): 1 - P(S)
    P(+|S): 0.8
    P(-|S): 1 - P(+|S)
    P(-|F): 0.9
    P(+|F): 1 - P(-|F)
    P(+): P(+|S) * P(S) + P(+|F) * P(F)
    P(-): P(-|F) * P(F) + P(-|S) * P(S)
    P(S|+): P(+|S) * P(S) / P(+)
    P(F|+): 1 - P(S|+)
    P(F|-): P(-|F) * P(F) / P(-)
    P(S|-): 1 - P(F|-)
    payoff_S: 5000000
    payoff_F: -1000000
type: decision
open:
    cost: fixed cost
    type: chance
    success:
        p: P(S)
        payoff: payoff_S
    failure:
        p: P(F)
        payoff: payoff_F
popup test:
    cost: popup cost
    type: chance
    test positive:
        p: P(+)
        type: decision
        open:
            cost: fixed cost
            type: chance
            success:
                p: P(S|+)
                payoff: payoff_S
            failure:
                p: P(F|+)
                payoff: payoff_F
        not open:
            payoff: 0

    test negative:
        p: P(-)
        type: decision
        open:
            cost: fixed cost
            type: chance
            success:
                p: P(S|-)
                payoff: payoff_S
            failure:
                p: P(F|-)
                payoff: payoff_F
        not open:
            payoff: 0
not open:
    payoff: 0
"""

In [4]:
tree = rsm.decide.dtree(full)
tree.summary(input=False, output=True)

Variable input values:
                            
fixed cost  2000000.00000000
popup cost   500000.00000000
P(S)              0.60000000
P(F)              0.40000000
P(+|S)            0.80000000
P(-|S)            0.20000000
P(-|F)            0.90000000
P(+|F)            0.10000000
P(+)              0.52000000
P(-)              0.48000000
P(S|+)            0.92307692
P(F|+)            0.07692308
P(F|-)            0.75000000
P(S|-)            0.25000000
payoff_S    5000000.00000000
payoff_F   -1000000.00000000

Initial decision tree:
                         Probability        Payoff         Cost     Type
 open new store                                                         
  ¦--open                                          2,000,000.00 decision
  ¦   ¦--success             60.00 %  5,000,000.00                chance
  ¦   °--failure             40.00 % -1,000,000.00                chance
  ¦--popup test                                      500,000.00 decision
  ¦   ¦--test positive

Opening right away is worth $600,000 (Example 1). Running the popup test first and opening only after a positive result is worth **$820,000**: run the popup test.

In [5]:
show(tree)

```mermaid
graph LR
    n1["$820,000.00"]
    n1 --- |"open"| n2
    n2(("$600,000.00"))
    n2 --- |"success: p=0.6000"| n3
    n3["$5,000,000.00"]
    n2 --- |"failure: p=0.4000"| n4
    n4["-$1,000,000.00"]
    n1 === |"popup test"| n5
    n5(("$820,000.00"))
    n5 --- |"test positive: p=0.5200"| n6
    n6["$2,538,461.54"]
    n6 === |"open"| n7
    n7(("$2,538,461.54"))
    n7 --- |"success: p=0.9231"| n8
    n8["$5,000,000.00"]
    n7 --- |"failure: p=0.0769"| n9
    n9["-$1,000,000.00"]
    n6 --- |"not open"| n10
    n10["$0.00"]
    n5 --- |"test negative: p=0.4800"| n11
    n11["$0.00"]
    n11 --- |"open"| n12
    n12(("-$1,500,000.00"))
    n12 --- |"success: p=0.2500"| n13
    n13["$5,000,000.00"]
    n12 --- |"failure: p=0.7500"| n14
    n14["-$1,000,000.00"]
    n11 === |"not open"| n15
    n15["$0.00"]
    n1 --- |"not open"| n16
    n16["$0.00"]
    classDef decision fill:#9ACD32,stroke:#333,stroke-width:1px;
    class n1,n6,n11 decision;
    classDef chance_with_cost fill:#FF8C00,stroke:#333,stroke-width:3px,stroke-dasharray:4 5;
    class n2,n5,n7,n12 chance_with_cost;
    %% chosen branches drawn with === (highlighted in final tree)
```